***

Preparing Workspace

***

In [ ]:


## Packages ---

import numpy as np
import pandas as pd
import getpass
from pathlib import Path
import os
import re
from datetime import date
import math
import seaborn as sns
import matplotlib.pyplot as plt
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.offline import plot
import plotly.subplots as sp
from plotly.subplots import make_subplots
pd.options.display.float_format = '{:.0f}'.format


## Set file paths ---

user = getpass.getuser()
path_users = Path.home()

path_git = path_users / 'Documents' / 'Projects' / 'Regional-Monitoring' / 'Indicator_Gen'
path_config0 = path_git / 'config'


## User defined functions ---

path_func = path_config0 / 'Functions.py'
with path_func.open("r") as f:
    exec(f.read())


path_plots = Path(r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring")
print('Export Location: ' + str(path_plots))


export=False



***

Production_1

***

In [ ]:
# Set Indicator
indicator = 'Production_1'
plot_name = 'total_units'


## Importing ---

file_name = f"{indicator} MPO SACOG Housing Permit Data.xlsx"
file_in = path_plots / 'Data' / file_name
sheet_name='MPO'
df_mpo = pd.read_excel(file_in, sheet_name=sheet_name)


## Organizing ---

df_plot = df_mpo.copy()


dict_plot_2023 = {
    'MPO': ['SACOG']
    , 'Year': ['2020 - 2035']
    , 'Total': [10406]
    }
df_plot_2023 = pd.DataFrame(dict_plot_2023)

df_plot = pd.concat([df_plot, df_plot_2023])


display(df_plot.head())


## Plotting ---

df_plot1 = df_plot[df_plot['Year'] != '2020 - 2035']
df_plot2 = df_plot[df_plot['Year'] == '2020 - 2035']

fig = make_subplots(rows=1, cols=2, column_widths=[0.9, 0.1], horizontal_spacing=0.1)
fig.add_trace(go.Bar(name='Total', x=df_plot1['Year'], y=df_plot1['Total'], marker_color='#1F45FC', width=0.75, showlegend=False), 1, 1)
fig.add_trace(go.Bar(name='Total', x=df_plot2['Year'], y=df_plot2['Total'], marker_color='#1F45FC', width=0.35, showlegend=False), 1, 2)

title = '<b>New Housing Units</b>  <br><sup>6-County Sacramento Region</sup>'
fig.update_yaxes(tick0=0, dtick=5000, range=[0, 26000], tickformat=',.0f')
fig.update_xaxes(tick0=0, dtick=1)
fig.update_traces(hovertemplate='%{y}')
fig.update_layout(barmode='stack')

fig['layout']['xaxis2']['title']='Projected Annual Average'

plot_agol(export=export)



***

Production_3

***

In [ ]:
# Set Indicator
indicator = 'Production_3'
plot_name = 'housing_gr_per_1k_residents'
year=2024


## Importing ---

file_name = f"{indicator} MPO DOF Summarized Data.xlsx"
file_in = path_plots / 'Data' / file_name
sheet_name='MPO'
df_mpo = pd.read_excel(file_in, sheet_name=sheet_name)


## Organizing ---

df_plot = df_mpo.copy()

df_plot = df_plot[df_plot['Year'] == year]
df_plot['Total_SF_MF_diff_per_1000_pop'] = round(df_plot['Total_SF_MF_diff_per_1000_pop'], 1)

conditions = [
    df_plot['MPO'] == 'SACOG'
    , df_plot['MPO']  == 'CVCOG'
    , df_plot['MPO']  == 'MTC'
    , df_plot['MPO']  == 'SCAG'
    , df_plot['MPO']  == 'SANDAG'
    , df_plot['MPO']  == 'Rest of CA'
    , df_plot['MPO']  == 'State Total'
]
choices = ['Sacramento', 'San Joaquin Valley', 'San Francisco Bay Area', 'Los Angeles', 'San Diego', 'Rest of State', 'State Total']
df_plot['Geography'] = np.select(conditions, choices, default = 'No')
df_plot = df_plot.sort_values('Total_SF_MF_diff_per_1000_pop', ascending=False)

df_plot = df_plot.reset_index(drop = True)

display(df_plot.head())


## Plotting ---


# fig = px.bar(df_plot, x='Geography', y='Total_SF_MF_diff_per_1000_pop')
# fig.update_traces(marker_color='#1E90FF')

color_map = {
         "State Total":"#1F45FC",
         "Rest of State": "#1E90FF",
         "San Diego": "#1E90FF",
         "Los Angeles": "#1E90FF",
         "San Francisco Bay Area": "#1E90FF",
         "San Joaquin Valley": "#1E90FF",
         "Sacramento": "#9DC209"
}

fig = px.bar(df_plot, x='Geography', y='Total_SF_MF_diff_per_1000_pop', color='Geography', color_discrete_map=color_map)


title = f'<b>Housing Growth Rate (New Units/1,000 Residents) by Region, {year}</b>'
fig.update_yaxes(dtick=0.5, range = [0,5.1])
fig.update_traces(hovertemplate="Housing Units Per 1k Residents: %{y}")
fig.update_layout(showlegend = False)

plot_agol(export=export)



***

Production_4

***

In [ ]:
# Set Indicator
indicator = 'Production_4'
plot_name = 'healthy_housing'


## Importing ---

file_name = f"{indicator} MPO SACOG Housing and DOF Population Data.xlsx"
file_in = path_plots / 'Data' / file_name
sheet_name='MPO'
df_mpo = pd.read_excel(file_in, sheet_name=sheet_name)


## Organizing ---

df_plot = df_mpo.copy()

df_plot['Estimated Number of Households Added'] = df_plot['Population Growth']/2.5

df_plot = pd.melt(df_plot, id_vars = ['MPO', 'Year'])
df_plot = df_plot[df_plot['variable'] != 'Population Growth']
display(df_plot.head())



## Plotting ---

# color_map = {
#     'Housing Growth':'#1E90FF'
#     , 'Healthy Housing Market Growth':'#9DC209'
# }

color_map = {
    'Number of New Housing Units Built':'#1E90FF'
    , 'Estimated Number of Households Added':'#FFA500'
}

fig = px.line(df_plot, x='Year', y='value', color='variable', markers=True, color_discrete_map=color_map)



title = '<b>Annual Housing Production Compared to Estimated Number of Households Added</b>  <br><sup>6-County Sacramento Region</sup>'
fig.update_yaxes(tick0=0, dtick=5000, range=[-5000, 31000], tickformat = ',.0f')
fig.update_xaxes(tick0=0, dtick=1, range=[2000.5, 2024.5])
fig.update_traces(hovertemplate='%{y}')
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=-0.15,xanchor="right", x=0.6))

plot_agol(export=export)


***

Production_6

***

In [ ]:
# Set Indicator
indicator = 'Production_6'
plot_name = 'product_type'


## Importing ---

file_name = f"{indicator} MPO SACOG Housing Permit Data.xlsx"
file_in = path_plots / 'Data' / file_name
sheet_name='MPO'
df_mpo = pd.read_excel(file_in, sheet_name=sheet_name)



## Organizing ---

df_plot = df_mpo.copy()



df_plot['SF_SFLL'] = df_plot['SF_SFLL'] + df_plot['SF_RR']
df_plot = df_plot.rename(columns = {'MF_total':'Multi Family', 'SF_SFLL':'Single Family Large Lot', 'SF_SFSL':'Single Family Small Lot'})
df_plot = pd.melt(df_plot, id_vars = ['MPO', 'Year'])
df_plot = df_plot[df_plot['variable'].isin(['Multi Family', 'Single Family Large Lot', 'Single Family Small Lot'])]


df_plot['Sort'] = pd.Categorical(df_plot['variable'], ['Multi Family', 'Single Family Small Lot', 'Single Family Large Lot'])
df_plot = df_plot.sort_values(['MPO', 'Year', 'Sort'], ascending = [True, False, True])
df_plot = df_plot.drop('Sort', axis = 1)
df_plot = df_plot.reset_index(drop=True)


dict_plot_2023 = {
    'MPO': ['SACOG', 'SACOG', 'SACOG']
    , 'variable': ['Single Family Large Lot', 'Single Family Small Lot', 'Multi Family']
    , 'Year': ['2020 - 2035', '2020 - 2035', '2020 - 2035']
    , 'value': [2630, 2673, 5104]
    }
df_plot_2023 = pd.DataFrame(dict_plot_2023)


df_plot = pd.concat([df_plot, df_plot_2023])

display(df_plot.head())



## Plotting ---

color_map = {
         "Multi Family":"#1F45FC",
         "Single Family Large Lot": "#9DC209",
         "Single Family Small Lot": "#1E90FF"
}


df_plot1 = df_plot[df_plot['Year'] != '2020 - 2035']
df_plot2 = df_plot[df_plot['Year'] == '2020 - 2035']

fig = make_subplots(rows=1, cols=2, column_widths=[0.9, 0.1], horizontal_spacing=0.1)
fig.add_trace(go.Bar(name='Multi Family'           , x=df_plot1[df_plot1['variable'] == 'Multi Family'           ]['Year'], y=df_plot1[df_plot1['variable'] == 'Multi Family'           ]['value'], marker_color='#1F45FC', width=0.75, legendgroup='C'), 1, 1)
fig.add_trace(go.Bar(name='Single Family Small Lot', x=df_plot1[df_plot1['variable'] == 'Single Family Small Lot']['Year'], y=df_plot1[df_plot1['variable'] == 'Single Family Small Lot']['value'], marker_color='#1E90FF', width=0.75, legendgroup='B'), 1, 1)
fig.add_trace(go.Bar(name='Single Family Large Lot', x=df_plot1[df_plot1['variable'] == 'Single Family Large Lot']['Year'], y=df_plot1[df_plot1['variable'] == 'Single Family Large Lot']['value'], marker_color='#9DC209', width=0.75, legendgroup='A'), 1, 1)

fig.add_trace(go.Bar(name='Multi Family'           , x=df_plot2[df_plot2['variable'] == 'Multi Family'           ]['Year'], y=df_plot2[df_plot2['variable'] == 'Multi Family'           ]['value'], marker_color='#1F45FC', width=0.45, legendgroup='C', showlegend=False), 1, 2)
fig.add_trace(go.Bar(name='Single Family Small Lot', x=df_plot2[df_plot2['variable'] == 'Single Family Small Lot']['Year'], y=df_plot2[df_plot2['variable'] == 'Single Family Small Lot']['value'], marker_color='#1E90FF', width=0.45, legendgroup='B', showlegend=False), 1, 2)
fig.add_trace(go.Bar(name='Single Family Large Lot', x=df_plot2[df_plot2['variable'] == 'Single Family Large Lot']['Year'], y=df_plot2[df_plot2['variable'] == 'Single Family Large Lot']['value'], marker_color='#9DC209', width=0.45, legendgroup='A', showlegend=False), 1, 2)

title = '<b>New Housing Units by Product Type</b>  <br><sup>6-County Sacramento Region</sup>'
fig.update_yaxes(tick0=0, dtick=5000, range=[0, 26000], tickformat=',.0f')
fig.update_xaxes(tick0=0, dtick=1)
fig.update_traces(hovertemplate='%{y}')
fig.update_layout(barmode='stack')
fig.update_layout(legend={'traceorder': 'reversed'})
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=-0.2,xanchor="right", x=0.65))

fig['layout']['xaxis2']['title']='Projected Annual Average'

plot_agol(export=export)



In [ ]:

# Set Indicator
indicator = 'Production_6'
plot_name = 'product_type_share'


## Importing ---

file_name = f"{indicator} MPO SACOG Housing Permit Data.xlsx"
file_in = path_plots / 'Data' / file_name
sheet_name='MPO'
df_mpo = pd.read_excel(file_in, sheet_name=sheet_name)



## Organizing ---

df_plot = df_mpo.copy()


df_plot['SF_SFLL'] = df_plot['SF_SFLL'] + df_plot['SF_RR']
df_plot = df_plot.rename(columns = {'MF_total':'Multi Family', 'SF_SFLL':'Single Family Large Lot', 'SF_SFSL':'Single Family Small Lot'})
df_plot = pd.melt(df_plot, id_vars = ['MPO', 'Year'])
df_plot = df_plot[df_plot['variable'].isin(['Multi Family', 'Single Family Large Lot', 'Single Family Small Lot'])]


df_plot['Sort'] = pd.Categorical(df_plot['variable'], ['Multi Family', 'Single Family Small Lot', 'Single Family Large Lot'])
df_plot = df_plot.sort_values(['MPO', 'Year', 'Sort'], ascending = [True, False, True])
df_plot = df_plot.drop('Sort', axis = 1)

df_plot['percentage'] = 100*df_plot['value'] / df_plot.groupby(['MPO', 'Year'])['value'].transform('sum')
df_plot['percentage'] = round(df_plot['percentage'], 1)


dict_plot_2023 = {
    'MPO': ['SACOG', 'SACOG', 'SACOG']
    , 'variable': ['Single Family Large Lot', 'Single Family Small Lot', 'Multi Family']
    , 'Year': ['2020 - 2035', '2020 - 2035', '2020 - 2035']
    , 'value': [2630, 2673, 5104]
    }
df_plot_2023 = pd.DataFrame(dict_plot_2023)


df_plot_2023['percentage'] = 100*df_plot_2023['value'] / df_plot_2023.groupby(['MPO', 'Year'])['value'].transform('sum')
df_plot_2023['percentage'] = round(df_plot_2023['percentage'], 1)


df_plot = pd.concat([df_plot, df_plot_2023])



display(df_plot.head())


## Plotting ---

color_map = {
         "Multi Family":"#1F45FC",
         "Single Family Large Lot": "#9DC209",
         "Single Family Small Lot": "#1E90FF"
}


df_plot1 = df_plot[df_plot['Year'] != '2020 - 2035']
df_plot2 = df_plot[df_plot['Year'] == '2020 - 2035']

fig = make_subplots(rows=1, cols=2, column_widths=[0.9, 0.1], horizontal_spacing=0.1)
fig.add_trace(go.Bar(name='Multi Family'           , x=df_plot1[df_plot1['variable'] == 'Multi Family'           ]['Year'], y=df_plot1[df_plot1['variable'] == 'Multi Family'           ]['percentage'], marker_color='#1F45FC', width=0.75, legendgroup='C'), 1, 1)
fig.add_trace(go.Bar(name='Single Family Small Lot', x=df_plot1[df_plot1['variable'] == 'Single Family Small Lot']['Year'], y=df_plot1[df_plot1['variable'] == 'Single Family Small Lot']['percentage'], marker_color='#1E90FF', width=0.75, legendgroup='B'), 1, 1)
fig.add_trace(go.Bar(name='Single Family Large Lot', x=df_plot1[df_plot1['variable'] == 'Single Family Large Lot']['Year'], y=df_plot1[df_plot1['variable'] == 'Single Family Large Lot']['percentage'], marker_color='#9DC209', width=0.75, legendgroup='A'), 1, 1)

fig.add_trace(go.Bar(name='Multi Family'           , x=df_plot2[df_plot2['variable'] == 'Multi Family'           ]['Year'], y=df_plot2[df_plot2['variable'] == 'Multi Family'           ]['percentage'], marker_color='#1F45FC', width=0.45, legendgroup='C', showlegend=False), 1, 2)
fig.add_trace(go.Bar(name='Single Family Small Lot', x=df_plot2[df_plot2['variable'] == 'Single Family Small Lot']['Year'], y=df_plot2[df_plot2['variable'] == 'Single Family Small Lot']['percentage'], marker_color='#1E90FF', width=0.45, legendgroup='B', showlegend=False), 1, 2)
fig.add_trace(go.Bar(name='Single Family Large Lot', x=df_plot2[df_plot2['variable'] == 'Single Family Large Lot']['Year'], y=df_plot2[df_plot2['variable'] == 'Single Family Large Lot']['percentage'], marker_color='#9DC209', width=0.45, legendgroup='A', showlegend=False), 1, 2)

title = '<b>New Housing Units by Product Type</b>  <br><sup>6-County Sacramento Region</sup>'
fig.update_yaxes(tick0=0, dtick=20, range = [0, 105], ticksuffix='%')
fig.update_xaxes(tick0=0, dtick=1)
fig.update_traces(hovertemplate='%{y}')
fig.update_layout(barmode='stack')
fig.update_layout(legend={'traceorder': 'reversed'})
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=-0.2,xanchor="right", x=0.65))

fig['layout']['xaxis2']['title']='Projected Annual Average'

plot_agol(export=export)


***

Production_2

***

In [ ]:
# Set Indicator
indicator = 'Production_2'
plot_name = 'multifamily_share'


## Importing ---

file_name = f"{indicator} MPO DOF Summarized Data.xlsx"
file_in = path_plots / 'Data' / file_name
sheet_name='MPO'
df_mpo = pd.read_excel(file_in, sheet_name=sheet_name)



## Organizing ---

df_plot = df_mpo.copy()

df_plot = df_plot[df_plot['Year'] > 2005]
df_plot = df_plot[['MPO', 'Year', 'MF_diff_pct']]
df_plot['MF_diff_pct'] = round(df_plot['MF_diff_pct'], 1)


conditions = [
                  df_plot['MPO'] == 'CVCOG'
                , df_plot['MPO'] == 'MTC'
                , df_plot['MPO'] == 'Rest of CA'
                , df_plot['MPO'] == 'SACOG'
                , df_plot['MPO'] == 'SANDAG'
                , df_plot['MPO'] == 'SCAG'
            ]

choices = ['San Joaquin Valley'
           , 'San Francisco Bay Area'
           , 'Rest of State'
           , 'Sacramento'
           , 'San Diego'
           , 'Los Angeles'
           ]

df_plot['Geography'] = np.select(conditions, choices, default = 'No')


df_plot['Sort'] = pd.Categorical(df_plot['Geography'], [
    'Sacramento'
    , 'San Joaquin Valley'
    , 'San Francisco Bay Area'
    , 'Los Angeles'
    , 'San Diego'
    , 'Rest of State'
    , 'State Total'
])

df_plot = df_plot.sort_values(['Sort', 'Year'], ascending = [True, True])
df_plot = df_plot.drop(['Sort'], axis = 1)

df_plot = df_plot.reset_index(drop = True)

display(df_plot.head())


## Plotting ---

color_map = {
    'Sacramento':'#9DC209'
    , 'San Joaquin Valley':"#7E587E"
    , 'San Francisco Bay Area':"#DC381F"
    , 'Los Angeles':"#1E90FF"
    , 'San Diego':"#FBB117"
    , 'Rest of State':"#1F45FC"
}


fig = px.line(df_plot, x='Year', y='MF_diff_pct', color='Geography', markers=True, color_discrete_map=color_map)


title = '<b>Multi-Family as Percentage of New Housing Units by CA Region, 5 year rolling averages</b>'
fig.update_yaxes(tick0=0, dtick=10, range = [0, 101], ticksuffix='%')
fig.update_xaxes(tick0=0, dtick=1, range = [2005.5, 2025.5])
fig.update_traces(hovertemplate='%{y}')
# fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=-0.2,xanchor="right", x=0.55))


plot_agol(export=export)


In [ ]:
# Set Indicator
indicator = 'Production_2'
plot_name = 'multifamily_share'


## Importing ---

file_name = f"{indicator} MPO DOF Summarized Data.xlsx"
file_in = path_plots / 'Data' / file_name
sheet_name='MPO'
df_mpo = pd.read_excel(file_in, sheet_name=sheet_name)



## Organizing ---

df_plot = df_mpo.copy()

df_plot = df_plot.sort_values(['MPO', 'Year'])
df_plot['MF_diff_pct_rolling_avg'] = df_plot.groupby(['MPO'])['MF_diff_pct'].transform(lambda x: x.rolling(5, 1).mean())
df_plot = df_plot[df_plot['Year'] > 2005]


df_plot = df_plot[['MPO', 'Year', 'MF_diff_pct_rolling_avg']]
df_plot['MF_diff_pct_rolling_avg'] = round(df_plot['MF_diff_pct_rolling_avg'], 1)


conditions = [
                  df_plot['MPO'] == 'CVCOG'
                , df_plot['MPO'] == 'MTC'
                , df_plot['MPO'] == 'Rest of CA'
                , df_plot['MPO'] == 'SACOG'
                , df_plot['MPO'] == 'SANDAG'
                , df_plot['MPO'] == 'SCAG'
            ]

choices = ['San Joaquin Valley'
           , 'San Francisco Bay Area'
           , 'Rest of State'
           , 'Sacramento'
           , 'San Diego'
           , 'Los Angeles'
           ]

df_plot['Geography'] = np.select(conditions, choices, default = 'No')


df_plot['Sort'] = pd.Categorical(df_plot['Geography'], [
    'Sacramento'
    , 'San Joaquin Valley'
    , 'San Francisco Bay Area'
    , 'Los Angeles'
    , 'San Diego'
    , 'Rest of State'
    , 'State Total'
])

df_plot = df_plot.sort_values(['Sort', 'Year'], ascending = [True, True])
df_plot = df_plot.drop(['Sort'], axis = 1)

df_plot = df_plot.reset_index(drop = True)

display(df_plot.head())


## Plotting ---

color_map = {
    'Sacramento':'#000000'
    , 'San Joaquin Valley':"#9DC209"
    , 'San Francisco Bay Area':"#DC381F"
    , 'Los Angeles':"#1E90FF"
    , 'San Diego':"#FBB117"
    , 'Rest of State':"#1F45FC"
}


fig = px.line(df_plot, x='Year', y='MF_diff_pct_rolling_avg', color='Geography', markers=True, color_discrete_map=color_map)


title = '<b>Multi-Family as Percentage of New Housing Units by CA Region, 5 year rolling averages</b>'
fig.update_yaxes(tick0=0, dtick=10, range = [0, 101], ticksuffix='%')
fig.update_xaxes(tick0=0, dtick=1, range = [2005.5, 2025.5])
fig.update_traces(hovertemplate='%{y}')
# fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=-0.2,xanchor="right", x=0.55))


plot_agol(export=export)


***

Location_1

***

In [ ]:


# Set Indicator
indicator = 'Location_1'
plot_name = 'community_type'


## Importing ---

file_name = f"{indicator} MPO SACOG Housing Permit Data.xlsx"
file_in = path_plots / 'Data' / file_name
sheet_name='MPO'
df_mpo = pd.read_excel(file_in, sheet_name=sheet_name)


## Organizing ---

df_plot = df_mpo.copy()

df_plot = df_plot[df_plot['Year']>=2008]
df_plot = pd.melt(df_plot, id_vars = ['MPO', 'Year'])


dict_plot_2023 = {
    'MPO': ['SACOG', 'SACOG', 'SACOG', 'SACOG', 'SACOG']
    , 'variable': ['Established Communities', 'Developing Communities', 'Centers and Corridors', 'Rural Residential', 'Agriculture Land']
    , 'Year': ['2020 - 2035', '2020 - 2035', '2020 - 2035', '2020 - 2035', '2020 - 2035']
    , 'value': [2990, 3110, 4213, 93, 0]
    }
df_plot_2023 = pd.DataFrame(dict_plot_2023)


conditions = [
                  df_plot['variable'] == 'COMTYP_CC'
                , df_plot['variable'] == 'COMTYP_EC'
                , df_plot['variable'] == 'COMTYP_DC'
                , df_plot['variable'] == 'COMTYP_RR'
                , df_plot['variable'] == 'COMTYP_AGNL_NA'
            ]

choices = ['Centers and Corridors'
           , 'Established Communities'
           , 'Developing Communities'
           , 'Rural Residential'
           , 'Agriculture Land'
           ]

df_plot['variable'] = np.select(conditions, choices, default = 'No')
df_plot = pd.concat([df_plot, df_plot_2023])


df_plot['Sort'] = pd.Categorical(df_plot['variable'], [
            'Established Communities'
            , 'Developing Communities'
            ,  'Centers and Corridors'
            , 'Rural Residential'
            , 'Agriculture Land'
        ])

df_plot = df_plot.sort_values(['Sort', 'Year'], ascending = [True, True])
df_plot = df_plot.drop(['Sort'], axis = 1)

df_plot = df_plot.dropna()
df_plot = df_plot.reset_index(drop = True)

display(df_plot.head())


## Plotting ---

color_map = {
    'Rural Residential':"#FBB117"
       , 'Established Communities':"#1F45FC"
       , 'Centers and Corridors':'#9DC209'
       , 'Developing Communities':"#1E90FF"
       , 'Agriculture Land':"#7E587E"
}

df_plot1 = df_plot[df_plot['Year'] != '2020 - 2035']
df_plot2 = df_plot[df_plot['Year'] == '2020 - 2035']

fig = make_subplots(rows=1, cols=2, column_widths=[0.9, 0.1], horizontal_spacing=0.1)
fig.add_trace(go.Bar(name='Established Communities', x=df_plot1[df_plot1['variable'] == 'Established Communities']['Year'], y=df_plot1[df_plot1['variable'] == 'Established Communities']['value'], marker_color='#1F45FC', width=0.75, legendgroup='A'), 1, 1)
fig.add_trace(go.Bar(name='Developing Communities' , x=df_plot1[df_plot1['variable'] == 'Developing Communities' ]['Year'], y=df_plot1[df_plot1['variable'] == 'Developing Communities' ]['value'], marker_color='#1E90FF', width=0.75, legendgroup='B'), 1, 1)
fig.add_trace(go.Bar(name='Centers and Corridors'  , x=df_plot1[df_plot1['variable'] == 'Centers and Corridors'  ]['Year'], y=df_plot1[df_plot1['variable'] == 'Centers and Corridors'  ]['value'], marker_color='#9DC209', width=0.75, legendgroup='C'), 1, 1)
fig.add_trace(go.Bar(name='Rural Residential'      , x=df_plot1[df_plot1['variable'] == 'Rural Residential'      ]['Year'], y=df_plot1[df_plot1['variable'] == 'Rural Residential'      ]['value'], marker_color='#FBB117', width=0.75, legendgroup='D'), 1, 1)
fig.add_trace(go.Bar(name='Agriculture Land'       , x=df_plot1[df_plot1['variable'] == 'Agriculture Land'       ]['Year'], y=df_plot1[df_plot1['variable'] == 'Agriculture Land'       ]['value'], marker_color='#7E587E', width=0.75, legendgroup='E'), 1, 1)

fig.add_trace(go.Bar(name='Established Communities', x=df_plot2[df_plot2['variable'] == 'Established Communities']['Year'], y=df_plot2[df_plot2['variable'] == 'Established Communities']['value'], marker_color='#1F45FC', width=0.45, legendgroup='A', showlegend=False), 1, 2)
fig.add_trace(go.Bar(name='Developing Communities' , x=df_plot2[df_plot2['variable'] == 'Developing Communities' ]['Year'], y=df_plot2[df_plot2['variable'] == 'Developing Communities' ]['value'], marker_color='#1E90FF', width=0.45, legendgroup='B', showlegend=False), 1, 2)
fig.add_trace(go.Bar(name='Centers and Corridors'  , x=df_plot2[df_plot2['variable'] == 'Centers and Corridors'  ]['Year'], y=df_plot2[df_plot2['variable'] == 'Centers and Corridors'  ]['value'], marker_color='#9DC209', width=0.45, legendgroup='C', showlegend=False), 1, 2)
fig.add_trace(go.Bar(name='Rural Residential'      , x=df_plot2[df_plot2['variable'] == 'Rural Residential'      ]['Year'], y=df_plot2[df_plot2['variable'] == 'Rural Residential'      ]['value'], marker_color='#FBB117', width=0.45, legendgroup='D', showlegend=False), 1, 2)
fig.add_trace(go.Bar(name='Agriculture Land'       , x=df_plot2[df_plot2['variable'] == 'Agriculture Land'       ]['Year'], y=df_plot2[df_plot2['variable'] == 'Agriculture Land'       ]['value'], marker_color='#7E587E', width=0.45, legendgroup='E', showlegend=False), 1, 2)

title = '<b>New Housing Units by Community Type</b>  <br><sup>6-County Sacramento Region</sup>'
fig.update_yaxes(tick0=0, dtick=2000, range = [0, 12100], tickformat = ',.0f')
fig.update_xaxes(tick0=0, dtick=1)
fig.update_traces(hovertemplate='%{y}')
fig.update_layout(barmode='stack')
fig.update_layout(legend={'traceorder': 'reversed'})

fig['layout']['xaxis2']['title']='Projected Annual Average'

plot_agol(export=export)



In [ ]:


# Set Indicator
indicator = 'Location_1'
plot_name = 'community_type_pct'



## Importing ---

file_name = f"{indicator} MPO SACOG Housing Permit Data.xlsx"
file_in = path_plots / 'Data' / file_name
sheet_name='MPO'
df_mpo = pd.read_excel(file_in, sheet_name=sheet_name)


## Organizing ---

df_plot = df_mpo.copy()

df_plot = df_plot[df_plot['Year']>=2008]
df_plot = pd.melt(df_plot, id_vars = ['MPO', 'Year'])


dict_plot_2023 = {
    'MPO': ['SACOG', 'SACOG', 'SACOG', 'SACOG', 'SACOG']
    , 'variable': ['Established Communities', 'Developing Communities', 'Centers and Corridors', 'Rural Residential', 'Agriculture Land']
    , 'Year': ['2020 - 2035', '2020 - 2035', '2020 - 2035', '2020 - 2035', '2020 - 2035']
    , 'value': [2990, 3110, 4213, 93, 0]
    }
df_plot_2023 = pd.DataFrame(dict_plot_2023)


conditions = [
                  df_plot['variable'] == 'COMTYP_CC'
                , df_plot['variable'] == 'COMTYP_EC'
                , df_plot['variable'] == 'COMTYP_DC'
                , df_plot['variable'] == 'COMTYP_RR'
                , df_plot['variable'] == 'COMTYP_AGNL_NA'
            ]

choices = ['Centers and Corridors'
           , 'Established Communities'
           , 'Developing Communities'
           , 'Rural Residential'
           , 'Agriculture Land'
           ]

df_plot['variable'] = np.select(conditions, choices, default = 'No')
df_plot = pd.concat([df_plot, df_plot_2023])


df_plot['Sort'] = pd.Categorical(df_plot['variable'], [
            'Established Communities'
            , 'Developing Communities'
            ,  'Centers and Corridors'
            , 'Rural Residential'
            , 'Agriculture Land'
        ])

df_plot = df_plot.sort_values(['Sort', 'Year'], ascending = [True, True])
df_plot = df_plot.drop(['Sort'], axis = 1)

df_plot = df_plot.dropna()

df_plot['percentage'] = 100*df_plot['value'] / df_plot.groupby(['Year'])['value'].transform('sum')
df_plot['percentage'] = round(df_plot['percentage'], 1)

df_plot = df_plot.reset_index(drop = True)

display(df_plot.head())


## Plotting ---

color_map = {
    'Rural Residential':"#FBB117"
       , 'Established Communities':"#1F45FC"
       , 'Centers and Corridors':'#9DC209'
       , 'Developing Communities':"#1E90FF"
       , 'Agriculture Land':"#7E587E"
}

df_plot1 = df_plot[df_plot['Year'] != '2020 - 2035']
df_plot2 = df_plot[df_plot['Year'] == '2020 - 2035']

fig = make_subplots(rows=1, cols=2, column_widths=[0.9, 0.1], horizontal_spacing=0.1)
fig.add_trace(go.Bar(name='Established Communities', x=df_plot1[df_plot1['variable'] == 'Established Communities']['Year'], y=df_plot1[df_plot1['variable'] == 'Established Communities']['percentage'], marker_color='#1F45FC', width=0.75, legendgroup='A'), 1, 1)
fig.add_trace(go.Bar(name='Developing Communities' , x=df_plot1[df_plot1['variable'] == 'Developing Communities' ]['Year'], y=df_plot1[df_plot1['variable'] == 'Developing Communities' ]['percentage'], marker_color='#1E90FF', width=0.75, legendgroup='B'), 1, 1)
fig.add_trace(go.Bar(name='Centers and Corridors'  , x=df_plot1[df_plot1['variable'] == 'Centers and Corridors'  ]['Year'], y=df_plot1[df_plot1['variable'] == 'Centers and Corridors'  ]['percentage'], marker_color='#9DC209', width=0.75, legendgroup='C'), 1, 1)
fig.add_trace(go.Bar(name='Rural Residential'      , x=df_plot1[df_plot1['variable'] == 'Rural Residential'      ]['Year'], y=df_plot1[df_plot1['variable'] == 'Rural Residential'      ]['percentage'], marker_color='#FBB117', width=0.75, legendgroup='D'), 1, 1)
fig.add_trace(go.Bar(name='Agriculture Land'       , x=df_plot1[df_plot1['variable'] == 'Agriculture Land'       ]['Year'], y=df_plot1[df_plot1['variable'] == 'Agriculture Land'       ]['percentage'], marker_color='#7E587E', width=0.75, legendgroup='E'), 1, 1)

fig.add_trace(go.Bar(name='Established Communities', x=df_plot2[df_plot2['variable'] == 'Established Communities']['Year'], y=df_plot2[df_plot2['variable'] == 'Established Communities']['percentage'], marker_color='#1F45FC', width=0.45, legendgroup='A', showlegend=False), 1, 2)
fig.add_trace(go.Bar(name='Developing Communities' , x=df_plot2[df_plot2['variable'] == 'Developing Communities' ]['Year'], y=df_plot2[df_plot2['variable'] == 'Developing Communities' ]['percentage'], marker_color='#1E90FF', width=0.45, legendgroup='B', showlegend=False), 1, 2)
fig.add_trace(go.Bar(name='Centers and Corridors'  , x=df_plot2[df_plot2['variable'] == 'Centers and Corridors'  ]['Year'], y=df_plot2[df_plot2['variable'] == 'Centers and Corridors'  ]['percentage'], marker_color='#9DC209', width=0.45, legendgroup='C', showlegend=False), 1, 2)
fig.add_trace(go.Bar(name='Rural Residential'      , x=df_plot2[df_plot2['variable'] == 'Rural Residential'      ]['Year'], y=df_plot2[df_plot2['variable'] == 'Rural Residential'      ]['percentage'], marker_color='#FBB117', width=0.45, legendgroup='D', showlegend=False), 1, 2)
fig.add_trace(go.Bar(name='Agriculture Land'       , x=df_plot2[df_plot2['variable'] == 'Agriculture Land'       ]['Year'], y=df_plot2[df_plot2['variable'] == 'Agriculture Land'       ]['percentage'], marker_color='#7E587E', width=0.45, legendgroup='E', showlegend=False), 1, 2)

title = '<b>New Housing Units by Community Type</b>  <br><sup>6-County Sacramento Region</sup>'
fig.update_yaxes(tick0=0, dtick=20, range = [0, 102], ticksuffix='%')
fig.update_xaxes(tick0=0, dtick=1)
fig.update_traces(hovertemplate='%{y}')
fig.update_layout(legend={'traceorder': 'reversed'})
fig.update_layout(barmode='stack')

fig['layout']['xaxis2']['title']='Projected Annual Average'

plot_agol(export=export)



***

Location_2a

***

In [ ]:
# Set Indicator
indicator = 'Location_2a'
plot_name = 'gz_share'

## Importing ---

file_name = f"{indicator} MPO SACOG Housing Permit Data.xlsx"
file_in = path_plots / 'Data' / file_name
sheet_name='MPO'
df_mpo = pd.read_excel(file_in, sheet_name=sheet_name)



## Organizing ---

df_plot = df_mpo.copy()
df_plot['GRZ_SHR'] = round(df_plot['GRZ_SHR'], 1)

display(df_plot.head())


## Plotting ---
fig = px.bar(df_plot, x='Year', y='GRZ_SHR')
fig.update_traces(marker_color='#1E90FF')


title = '<b>Green Zone Housing Growth as Percent of New Units</b>  <br><sup>6-County Sacramento Region</sup>'
fig.update_layout(showlegend=False)
fig.update_yaxes(tick0=0, dtick=5, ticksuffix='%', range = [0, 32])
fig.update_xaxes(tick0=0, dtick=1)
fig.update_traces(hovertemplate='%{y}')


plot_agol(export=export)


In [ ]:
# Set Indicator
indicator = 'Location_2a'
plot_name = 'gz_share_total'

## Importing ---

file_name = f"{indicator} MPO SACOG Housing Permit Data.xlsx"
file_in = path_plots / 'Data' / file_name
sheet_name='MPO'
df_mpo = pd.read_excel(file_in, sheet_name=sheet_name)


## Organizing ---

df_plot = df_mpo.copy()


file_name = "Production_1 MPO SACOG Housing Permit Data.xlsx"
file_in = path_plots / 'Data' / file_name
sheet_name='MPO'
df_total = pd.read_excel(file_in, sheet_name=sheet_name)


df_plot = df_mpo.merge(df_total, on=['MPO', 'Year'], how='left')
df_plot['Total_green_zone'] = round(df_plot['Total']*df_plot['GRZ_SHR']/100)


display(df_plot.head())


## Plotting ---
fig = px.bar(df_plot, x='Year', y='Total_green_zone')
fig.update_traces(marker_color='#1E90FF')


title = '<b>Green Zone Housing Growth as Percent of New Units</b>  <br><sup>6-County Sacramento Region</sup>'
fig.update_yaxes(tick0=0, dtick=500, tickformat = ',.0f', range = [0, 3550])
fig.update_xaxes(tick0=0, dtick=1)
fig.update_traces(hovertemplate='%{y}')
fig.update_layout(showlegend = False)


plot_agol(export=export)



***

Location_2b

***

In [ ]:
# # Set Indicator
# indicator_name = 'Location_2b'
# plot_name = 'gz_product_type'
# export = False



# ## Importing ---

# file_name = f"{indicator_name} MPO SACOG Housing Permit Data.xlsx"
# df_mpo = pd.read_excel(os.path.join(path_plots, 'Data', file_name), sheet_name = 'MPO')



# ## Organizing ---


# df_plot = df_mpo.copy()

# df_plot = pd.melt(df_plot, id_vars = ['MPO', 'Year'])
# df_plot = df_plot.sort_values(['MPO', 'Year', 'variable'], ascending = [True, False, False])
# df_plot = df_plot[~df_plot['variable'].isin(['GRZ_SF', 'GRZ_MF2t4', 'GRZ_MF5'])]
# conditions = [
#     df_plot['variable'].isin(['GRZ_SFSL_NO', 'GRZ_RR_SFLL_NO', 'GRZ_MF_NO'])
#     , df_plot['variable'].isin(['GRZ_SFSL', 'GRZ_SFLL', 'GRZ_MF'])
# ]
# choices = ['<b>Not in Green Zone</b>', '<b>Green Zone</b>']
# df_plot['green_zone'] = np.select(conditions, choices, default = 'No')

# conditions = [
#     df_plot['variable'].isin(['GRZ_SFSL_NO', 'GRZ_SFSL'])
#     , df_plot['variable'].isin(['GRZ_RR_SFLL_NO', 'GRZ_SFLL'])
#     , df_plot['variable'].isin(['GRZ_MF_NO', 'GRZ_MF'])
# ]
# choices = ['Single Family Small Lot', 'Single Family Large Lot', 'Multi Family']
# df_plot['product_type'] = np.select(conditions, choices, default = 'No')
# df_plot = df_plot[['MPO', 'Year', 'green_zone', 'product_type', 'value']]
# df_plot = df_plot.groupby(['MPO', 'Year', 'green_zone', 'product_type'], as_index = False)['value'].sum()

# df_plot['Sort'] = pd.Categorical(df_plot['product_type'], [
#          "Multi Family",
#          "Single Family Small Lot",
#          "Single Family Large Lot"
#            ])


# df_plot = df_plot.sort_values(['MPO', 'Year', 'green_zone', 'Sort'], ascending = [True, False, True, True])
# df_plot = df_plot.drop(['Sort'], axis = 1)
# df_plot = df_plot[df_plot['Year'] == 2023]
# display(df_plot.head())



# ## Plotting ---

# color_map = {
#          "Multi Family":"#1F45FC",
#          "Single Family Large Lot": "#9DC209",
#          "Single Family Small Lot": "#1E90FF"
# }


# fig = px.bar(df_plot, x='Year', y='value', color='product_type', color_discrete_map=color_map, facet_col='green_zone', barmode='group')


# # title = '<b>New Housing Built in Green Zones Compared to Rest of Region, by Product Type, 2023</b>  <br><sup>6-County Sacramento Region</sup>'
# title = '<b>New Housing Built in Green Zones by Product Type, 2023</b>  <br><sup>6-County Sacramento Region</sup>'

# fig.update_yaxes(tick0=0, dtick=1000, range = [0, 5250], tickformat = ',.0f')
# fig.update_xaxes(showticklabels=False)
# fig.update_traces(hovertemplate='%{y:,.0f}')

# for a in fig.layout.annotations:
#     a.text = a.text.split("=")[1]
# fig['layout']['xaxis2']['title']['text']=''
# for annotation in fig['layout']['annotations']:
#     annotation['y'] = -0.1
# fig.update_annotations(font_size=14)

# plot_agol(export=export)


In [ ]:
# Set Indicator
indicator = 'Location_2b'
plot_name = 'gz_product_type'
year = 2024


## Importing ---

file_name = f"{indicator} MPO SACOG Housing Permit Data.xlsx"
file_in = path_plots / 'Data' / file_name
sheet_name='MPO'
df_mpo = pd.read_excel(file_in, sheet_name=sheet_name)



## Organizing ---


df_plot = df_mpo.copy()

df_plot = pd.melt(df_plot, id_vars = ['MPO', 'Year'])
df_plot = df_plot.sort_values(['MPO', 'Year', 'variable'], ascending = [True, False, False])
df_plot = df_plot[~df_plot['variable'].isin(['GRZ_SF', 'GRZ_MF2t4', 'GRZ_MF5'])]
conditions = [
    df_plot['variable'].isin(['GRZ_SFSL_NO', 'GRZ_RR_SFLL_NO', 'GRZ_MF_NO'])
    , df_plot['variable'].isin(['GRZ_SFSL', 'GRZ_SFLL', 'GRZ_MF'])
]
choices = ['<b>Not in Green Zone</b>', '<b>Green Zone</b>']
df_plot['green_zone'] = np.select(conditions, choices, default = 'No')

conditions = [
    df_plot['variable'].isin(['GRZ_SFSL_NO', 'GRZ_SFSL'])
    , df_plot['variable'].isin(['GRZ_RR_SFLL_NO', 'GRZ_SFLL'])
    , df_plot['variable'].isin(['GRZ_MF_NO', 'GRZ_MF'])
]
choices = ['Single Family Small Lot', 'Single Family Large Lot', 'Multi Family']
df_plot['product_type'] = np.select(conditions, choices, default = 'No')
df_plot = df_plot[['MPO', 'Year', 'green_zone', 'product_type', 'value']]
df_plot = df_plot.groupby(['MPO', 'Year', 'green_zone', 'product_type'], as_index = False)['value'].sum()


df_plot['Sort'] = pd.Categorical(df_plot['product_type'], [
         "Multi Family",
         "Single Family Small Lot",
         "Single Family Large Lot"
           ])


df_plot = df_plot.sort_values(['MPO', 'Year', 'green_zone', 'Sort'], ascending = [True, False, True, True])
df_plot = df_plot.drop(['Sort'], axis = 1)

df_plot = df_plot[df_plot['Year'] == year]
display(df_plot.head())



## Plotting ---

color_map = {
         "Multi Family":"#1F45FC",
         "Single Family Large Lot": "#9DC209",
         "Single Family Small Lot": "#1E90FF"
}


fig = px.pie(df_plot, values='value', names='product_type', facet_col='green_zone', color = 'product_type', color_discrete_map=color_map)
fig.update_traces(textfont_size=14)

# title = '<b>New Housing Built in Green Zones Compared to Rest of Region, by Product Type, 2023</b>  <br><sup>6-County Sacramento Region</sup>'
title = f'<b>New Housing Built in Green Zones by Product Type, {year}</b>  <br><sup>6-County Sacramento Region</sup>'

fig.update_yaxes(tick0=0, dtick=1000, range = [0, 5250], tickformat = ',.0f')
fig.update_traces(hovertemplate='%{value:,.0f}')
for a in fig.layout.annotations:
    a.text = a.text.split("=")[1]
for annotation in fig['layout']['annotations']:
    annotation['y'] = -0.1
fig.update_annotations(font_size=14)

plot_agol(export=export)


***

Location_3

***

In [ ]:
# Set Indicator
indicator = 'Location_3'
plot_name = 'acres'


## Importing ---

file_name = f"{indicator}.xlsx"
file_in = path_plots / 'Data' / file_name
sheet_name='Data'
df_mpo = pd.read_excel(file_in, sheet_name=sheet_name)




## Organizing ---

df_plot = df_mpo.copy()

df_temp = df_plot.copy()
df_temp = df_temp[df_temp['FMMP_Type'].isin(['Farmland of Local Potential', 'Farmland of Local Importance'])]
df_temp = df_temp.groupby(['YEAR_'], as_index = False)[['Units of new housing by location type', 'Acres of new housing by location type']].sum()
df_temp['FMMP_Type'] = 'Farmland of Local Importance/Potential'
df_plot = pd.concat([df_plot, df_temp])

df_plot = df_plot[df_plot['FMMP_Type'].isin(['Unique Farmland', 'Farmland of Statewide Importance', 'Prime Farmland', 'Farmland of Local Importance/Potential'])]


df_plot['Sort'] = pd.Categorical(df_plot['FMMP_Type'], [
     'Farmland of Local Importance/Potential'
     , 'Farmland of Statewide Importance'
     , 'Prime Farmland'
     , 'Unique Farmland'
           ])

df_plot = df_plot.sort_values(['Sort', 'YEAR_'], ascending = [True, True])
df_plot = df_plot.drop(['Sort'], axis = 1)
df_plot = df_plot[df_plot['YEAR_'] > 2009]

df_plot = df_plot.reset_index(drop = True)

display(df_plot.head())


## Plotting ---

color_map = {
    'Unique Farmland':"#FBB117"
       , 'Farmland of Statewide Importance':"#1E90FF"
       , 'Prime Farmland':'#9DC209'
       , 'Farmland of Local Importance/Potential':"#1F45FC"
}


fig = px.bar(df_plot, x='YEAR_', y='Acres of new housing by location type', color='FMMP_Type', color_discrete_map=color_map)


title = '<b>Annual Acres of Important Farmland Urbanized by Housing</b>  <br><sup>6-County Sacramento Region</sup>'
fig.update_yaxes(tick0=0, dtick=500, range = [0, 4100], tickformat = ',.0f')
fig.update_xaxes(tick0=0, dtick=1)
fig.update_traces(hovertemplate='%{y}')
fig.update_layout(legend={'traceorder': 'reversed'})


plot_agol(export=export)


***

Location_4

***

In [ ]:
# Set Indicator
indicator = 'Location_4'
plot_name = 'fire_flood'


## Importing ---

file_name = f"{indicator} Fire and Flood.xlsx"
file_in = path_plots / 'Data' / file_name
sheet_name='in'
df_mpo = pd.read_excel(file_in, sheet_name=sheet_name)



## Organizing ---

df_plot = df_mpo.copy()

df_plot.columns = ['Year', 'Flood Zone', 'Fire Zone']
df_plot = pd.melt(df_plot, id_vars = 'Year')
df_plot = df_plot.reset_index()

display(df_plot.head())

## Plotting ---
color_map = {
    'Flood Zone':"#1E90FF"
       , 'Fire Zone':"#DC381F"
}


fig = px.bar(df_plot, x='Year', y='value', color='variable', color_discrete_map=color_map)


title = '<b>Housing Units Built in 100 Year Flood Plain and Fire Severity Zones</b>  <br><sup>6-County Sacramento Region</sup>'
fig.update_yaxes(tick0=0, dtick=1000, range = [0, 5100], tickformat = ',.0f')
fig.update_xaxes(tick0=0, dtick=1)
fig.update_traces(hovertemplate='%{y}')
fig.update_layout(legend={'traceorder': 'reversed'})


plot_agol(export=export)


***

Production_7

***

In [ ]:


# Set Indicator
indicator = 'Production_7'
plot_name = 'total_adu'


## Importing ---

file_name = f"{indicator} MPO SACOG Housing Permit Data.xlsx"
file_in = path_plots / 'Data' / file_name
sheet_name='MPO'
df_mpo = pd.read_excel(file_in, sheet_name=sheet_name)


## Organizing ---

df_plot = df_mpo.copy()
display(df_plot.head())


## Plotting ---
fig = px.bar(df_plot, x='Year', y='ADU_total')
fig.update_traces(marker_color='#1E90FF')


title = '<b>New Accessory Dwelling Units</b>  <br><sup>6-County Sacramento Region</sup>'
fig.update_yaxes(tick0=0, dtick=100, tickformat = ',.0f', range = [0, 850])
fig.update_xaxes(tick0=0, dtick=1)
fig.update_traces(hovertemplate='%{y}')
fig.update_layout(showlegend = False)

plot_agol(export=export)



***

Cost_4

***

In [ ]:
# Set Indicator
indicator = 'Cost_4'
plot_name = 'rhna_income'


## Importing ---

file_name = f"{indicator} MPO_SACOG HCD Summarized Data.xlsx"
file_in = path_plots / 'Data' / file_name
sheet_name='MPO'
df_mpo = pd.read_excel(file_in, sheet_name=sheet_name)



## Organizing ---

df_plot = df_mpo.copy()

df_plot = df_plot.rename(columns = {'CO_VLI':'Very Low Income', 'CO_LI':'Low Income', 'CO_MI':'Moderate Income', 'CO_AMI':'Above Moderate Income'})
df_plot = pd.melt(df_plot, id_vars = ['Year'])
df_plot = df_plot[df_plot['variable'].isin(['Very Low Income', 'Low Income', 'Moderate Income', 'Above Moderate Income'])]


df_plot['Sort'] = pd.Categorical(df_plot['variable'], ['Very Low Income', 'Low Income', 'Moderate Income', 'Above Moderate Income'])
df_plot = df_plot.sort_values(['Year', 'Sort'], ascending = [False, True])
df_plot = df_plot.drop('Sort', axis = 1)
df_plot = df_plot.reset_index(drop=True)
df_plot['Percentage'] = round( (df_plot['value'] / df_plot.groupby('Year')['value'].transform('sum')) * 100, 1)
df_plot = df_plot.drop('value',axis=1)

dict_plot_6th = {
    'variable': ['Very Low Income', 'Low Income', 'Moderate Income', 'Above Moderate Income']
    , 'Year': ['6th Cycle RHNA', '6th Cycle RHNA', '6th Cycle RHNA', '6th Cycle RHNA']
    , 'Percentage': [25, 15, 18, 42]
    }
df_plot_6th = pd.DataFrame(dict_plot_6th)


df_plot = pd.concat([df_plot, df_plot_6th])

display(df_plot.head())



## Plotting ---

color_map = {
         "Very Low Income":"#1F45FC",
         "Low Income": "#1E90FF",
         "Moderate Income": "#9DC209",
         "Above Moderate Income":"#FBB117"
}


df_plot1 = df_plot[df_plot['Year'] != '6th Cycle RHNA']
df_plot2 = df_plot[df_plot['Year'] == '6th Cycle RHNA']

fig = make_subplots(rows=1, cols=2, column_widths=[0.9, 0.1], horizontal_spacing=0.1)
fig.add_trace(go.Bar(name='Very Low Income'      , x=df_plot1[df_plot1['variable'] == 'Very Low Income'      ]['Year'], y=df_plot1[df_plot1['variable'] == 'Very Low Income'      ]['Percentage'], marker_color='#1F45FC', width=0.75, legendgroup='A'), 1, 1)
fig.add_trace(go.Bar(name='Low Income'           , x=df_plot1[df_plot1['variable'] == 'Low Income'           ]['Year'], y=df_plot1[df_plot1['variable'] == 'Low Income'           ]['Percentage'], marker_color='#1E90FF', width=0.75, legendgroup='B'), 1, 1)
fig.add_trace(go.Bar(name='Moderate Income'      , x=df_plot1[df_plot1['variable'] == 'Moderate Income'      ]['Year'], y=df_plot1[df_plot1['variable'] == 'Moderate Income'      ]['Percentage'], marker_color='#9DC209', width=0.75, legendgroup='C'), 1, 1)
fig.add_trace(go.Bar(name='Above Moderate Income', x=df_plot1[df_plot1['variable'] == 'Above Moderate Income']['Year'], y=df_plot1[df_plot1['variable'] == 'Above Moderate Income']['Percentage'], marker_color='#FBB117', width=0.75, legendgroup='D'), 1, 1)

fig.add_trace(go.Bar(name='Very Low Income'      , x=df_plot2[df_plot2['variable'] == 'Very Low Income'      ]['Year'], y=df_plot2[df_plot2['variable'] == 'Very Low Income'      ]['Percentage'], marker_color='#1F45FC', width=0.45, legendgroup='A', showlegend=False), 1, 2)
fig.add_trace(go.Bar(name='Low Income'           , x=df_plot2[df_plot2['variable'] == 'Low Income'           ]['Year'], y=df_plot2[df_plot2['variable'] == 'Low Income'           ]['Percentage'], marker_color='#1E90FF', width=0.45, legendgroup='B', showlegend=False), 1, 2)
fig.add_trace(go.Bar(name='Moderate Income'      , x=df_plot2[df_plot2['variable'] == 'Moderate Income'      ]['Year'], y=df_plot2[df_plot2['variable'] == 'Moderate Income'      ]['Percentage'], marker_color='#9DC209', width=0.45, legendgroup='C', showlegend=False), 1, 2)
fig.add_trace(go.Bar(name='Above Moderate Income', x=df_plot2[df_plot2['variable'] == 'Above Moderate Income']['Year'], y=df_plot2[df_plot2['variable'] == 'Above Moderate Income']['Percentage'], marker_color='#FBB117', width=0.45, legendgroup='D', showlegend=False), 1, 2)

title = '<b>New Housing Units by Income Category</b>  <br><sup>6-County Sacramento Region</sup>'
fig.update_yaxes(tick0=0, dtick=20, range = [0, 102], ticksuffix='%')
fig.update_xaxes(tick0=0, dtick=1)
fig.update_traces(hovertemplate='%{y}')
fig.update_layout(legend={'traceorder': 'reversed'})
fig.update_layout(barmode='stack')


plot_agol(export=export)

